In [7]:
import json
import pennylane as qp
import pennylane.numpy as np
def swap_test(n_wires_state,build_state_x,build_state_y): 
    """
    A function that returns the value of the SWAP test using 2*n_wires_state+1 qubits.
    
    Args: 
        n_wires_state (int): Number of wires needed to encode the quantum state.
        build_state_x (function): Function that applies the unitary gate to build the quantum state x_vector on some wires.  
        build_state_y (function): Function that applies the unitary gate to build the quantum state y_vector on some wires. 

    Returns: 
        (np.tensor): A numpy tensor of 1 element corresponding to the expectation value of the Pauli Z operator in the first qubit, which corresponds to the difference P(0)-P(1).
    """

    x_wires = list(range(1, n_wires_state + 1))
    y_wires = list(range(n_wires_state + 1, 2 * n_wires_state + 1))

    build_state_x(x_wires)
    build_state_y(y_wires)

    qp.Hadamard(wires=0)
    for xw, yw in zip(x_wires, y_wires):
        qp.CSWAP(wires=[0, xw, yw])
    qp.Hadamard(wires=0)

    return qp.expval(qp.PauliZ(0))

In [8]:
def calculate_distance(x_norm,y_norm,n_wires_state,build_state_x,build_state_y):
    """
    A function that returns the distance between the two vectors x_vector and y_vector using the function swap_test.

    Args:
        x_norm (float): Norm of x_vector.
        y_norm (float): Norm of x_vector.       
        n_wires_state (int): Number of wires needed to encode the quantum state.  
        build_state_x (function): Function that applies the unitary gate to build the quantum state x_vector on some wires.  
        build_state_y (function): Function that applies the unitary gate to build the quantum state y_vector on some wires.  

    Returns: 
        (np.tensor): Value of the difference |x_vector-y_vector|
        
    """
    dev=qp.device('default.qubit',wires=2*n_wires_state+1)
    qnode=qp.QNode(swap_test,dev)
    output_swap_test = qnode(n_wires_state,build_state_x, build_state_y) 

    # Use the output from qnode to compute the distance

In [9]:
def swap_test_less_qubits(x_norm,y_norm,n_wires_state,build_state_x,build_state_y):
    """
    A function that returns the value of the SWAP test using n_wires_state+3 qubits

    Args:
        x_norm (float): Norm of x_vector.
        y_norm (float): Norm of x_vector.       
        n_wires_state (int): Number of wires needed to encode the quantum state.  
        build_state_x (function): Function that applies the unitary gate to build the quantum state x_vector on some wires.  
        build_state_y (function): Function that applies the unitary gate to build the quantum state y_vector on some wires.   

    Returns: 
        (np.tensor): A numpy tensor of 1 element corresponding to the expectation value of the Pauli Z operator in the first qubit, which corresponds to the difference P(0)-P(1).
        
    """

    # Put your code here #

    # Return some measurement  

In [10]:
def calculate_distance_less_qubits(x_norm,y_norm,n_wires_state,build_state_x,build_state_y):
    """
    A function that returns the distance between the two vectors x_vector and y_vector using the function swap_test_less_qubits.

    Args:
        x_norm (float): Norm of x_vector.
        y_norm (float): Norm of x_vector.       
        n_wires_state (int): Number of wires needed to encode the quantum state.  
        build_state_x (function): Function that applies the unitary gate to build the quantum state x_vector on some wires.  
        build_state_y (function): Function that applies the unitary gate to build the quantum state y_vector on some wires.  

    Returns: 
        (np.tensor): Value of the difference |x_vector-y_vector|
        
    """
    dev=qp.device('default.qubit',wires=n_wires_state+3)
    qnode=qp.QNode(swap_test_less_qubits,dev)
    output_swap_test_less_qubits = qnode(x_norm,y_norm,n_wires_state,build_state_x, build_state_y)  

    # Use the output from qnode to compute the distance
    


In [11]:
# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    inputs=ins[2:]

    def build_state_x(wires):
        return qp.AmplitudeEmbedding(ins[0],wires=wires,pad_with=0,normalize=True)
    def build_state_y(wires):
        return qp.AmplitudeEmbedding(ins[1],wires=wires,pad_with=0,normalize=True)
    inputs.append(build_state_x)
    inputs.append(build_state_y)
    output_1 = calculate_distance(*inputs)
    output_2 = calculate_distance_less_qubits(*inputs)
    output = [output_1,output_2]

    return json.dumps([float(v) for v in output])

def check(solution_output: str, expected_output: str) -> None:
    solution_output=json.loads(solution_output)
    expected_output=json.loads(expected_output)
    assert np.allclose(
        np.array(solution_output), np.array(expected_output), atol=1e-5
    ), "Your function does not give the correct distance."

    dev=qp.device('default.qubit',wires=100)
    swap_test_qnode=qp.QNode(swap_test,dev)
    swap_test_less_qubits_qnode=qp.QNode(swap_test_less_qubits,dev)
    # For testing your solution we build the QNodes with placeholders
    def build_state_x(wires):
        return qp.AmplitudeEmbedding(np.array([1,0]),wires=wires,pad_with=0)
    
    
    gates_1 = qp.specs(swap_test_qnode)(1, build_state_x, build_state_x).resources.gate_types
    gates_2 = qp.specs(swap_test_less_qubits_qnode)(1, 1, 1, build_state_x, build_state_x).resources.gate_types
   
    names_1 = list(gates_1.keys())
    names_2 = list(gates_2.keys())
    assert not any('Adjoint' in name for name in names_1),"Your circuit is using the adjoint function, you don't need that for the SWAP test!"
    assert names_1.count('CSWAP') > 0, "Your circuit needs to implement at least 1 CSWAP!"
    assert not any('Adjoint' in name for name in names_2),"Your circuit is using the adjoint function, you don't need that for the SWAP test!"
    assert names_2.count('CSWAP') > 0, "Your circuit needs to implement at least 1 CSWAP!"

In [12]:
# These are the public test cases
test_cases = [
    ('[[1, 1], [8, 6], 1.4142135623730951, 10.0, 1]', '8.602325267042627'),
    ('[[2, 5, 9, 7], [5, 6, 3, 9], 12.609520212918492, 12.288205727444508, 2]', '7.0710678118654755')
]
# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")

Running test case 0 with input '[[1, 1], [8, 6], 1.4142135623730951, 10.0, 1]'...
Runtime Error. A quantum function must return either a single measurement, or a nonempty sequence of measurements.
Running test case 1 with input '[[2, 5, 9, 7], [5, 6, 3, 9], 12.609520212918492, 12.288205727444508, 2]'...
Runtime Error. A quantum function must return either a single measurement, or a nonempty sequence of measurements.
